# Combined Label Generation (2015-2016 + 2017-2018)

In [1]:
import pandas as pd
import numpy as np
import os, sys

sys.path.append("..")

PROCESSED_DIR = "../data/processed"
INTERIM_DIR = "../data/interim"
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [2]:
components = [
    "DEMO", "BMX", "GHB", "GLU", "BPX", "BIOPRO",
    "DIQ", "KIQ_U", "SMQ", "ALQ", "PAQ", "DBQ",
    "MCQ", "BPQ", "CBC", "HDL", "TCHOL", "TRIGLY",
    "INS", "HSCRP", "ALB_CR", "HSQ"
]

combined = {}

for comp in components:
    # 2013-2014 (cycle=0)
    path_2013 = os.path.join(INTERIM_DIR, f"{comp}_2013_verified.csv")
    # 2015-2016 (cycle=1)
    path_2015 = os.path.join(INTERIM_DIR, f"{comp}_2015_verified.csv")
    # 2017-2018 (cycle=2)
    path_2017 = os.path.join(INTERIM_DIR, f"{comp}_verified.csv")

    dfs = []
    if os.path.exists(path_2013):
        df_2013 = pd.read_csv(path_2013)
        df_2013["cycle"] = 0
        # if HSCRP is missing in 2013, add column with NaN
        if comp == "HSCRP":
            df_2013["LBXHSCRP"] = np.nan
        dfs.append(df_2013)
        print(f"{comp}: loaded 2013 {df_2013.shape}")
    else:
        print(f"{comp}: 2013 file missing")

    if os.path.exists(path_2015):
        df_2015 = pd.read_csv(path_2015)
        df_2015["cycle"] = 1
        dfs.append(df_2015)
        print(f"{comp}: loaded 2015 {df_2015.shape}")
    else:
        print(f"{comp}: 2015 file missing")

    if os.path.exists(path_2017):
        df_2017 = pd.read_csv(path_2017)
        df_2017["cycle"] = 2
        dfs.append(df_2017)
        print(f"{comp}: loaded 2017 {df_2017.shape}")
    else:
        print(f"{comp}: 2017 file missing")

    if dfs:
        combined[comp] = pd.concat(dfs, ignore_index=True)
        print(f"{comp}: combined {combined[comp].shape}")
    else:
        print(f"{comp}: no data loaded")

DEMO: loaded 2013 (10175, 48)
DEMO: loaded 2015 (9971, 48)
DEMO: loaded 2017 (9254, 47)
DEMO: combined (29400, 52)
BMX: loaded 2013 (9813, 27)
BMX: loaded 2015 (9544, 27)
BMX: loaded 2017 (8704, 22)
BMX: combined (28061, 29)
GHB: loaded 2013 (6979, 3)
GHB: loaded 2015 (6744, 3)
GHB: loaded 2017 (6401, 3)
GHB: combined (20124, 3)
GLU: loaded 2013 (3329, 7)
GLU: loaded 2015 (3191, 5)
GLU: loaded 2017 (3036, 5)
GLU: combined (9556, 7)
BPX: loaded 2013 (9813, 24)
BPX: loaded 2015 (9544, 22)
BPX: loaded 2017 (8704, 22)
BPX: combined (28061, 24)
BIOPRO: loaded 2013 (6979, 39)
BIOPRO: loaded 2015 (6744, 39)
BIOPRO: loaded 2017 (6401, 42)
BIOPRO: combined (20124, 42)
DIQ: loaded 2013 (9770, 55)
DIQ: loaded 2015 (9575, 55)
DIQ: loaded 2017 (8897, 55)
DIQ: combined (28242, 55)
KIQ_U: loaded 2013 (5769, 17)
KIQ_U: loaded 2015 (5719, 17)
KIQ_U: loaded 2017 (5569, 17)
KIQ_U: combined (17057, 18)
SMQ: loaded 2013 (7168, 33)
SMQ: loaded 2015 (7001, 43)
SMQ: loaded 2017 (6724, 38)
SMQ: combined (20893

In [3]:
df = combined["DEMO"].copy()

merge_order = [
    "BMX", "GHB", "GLU", "BPX", "BIOPRO",
    "DIQ", "KIQ_U", "SMQ", "ALQ", "PAQ", "DBQ",
    "MCQ", "BPQ", "CBC", "HDL", "TCHOL", "TRIGLY",
    "INS", "HSCRP", "ALB_CR", "HSQ"
]

for ds_name in merge_order:
    if ds_name in combined:
        df = df.merge(combined[ds_name], on=["SEQN", "cycle"], how="inner")
        print(f"Merged {ds_name}: {df.shape}")

print(f"\nFinal master shape: {df.shape}")
print("Duplicate SEQN+cycle:", df[["SEQN", "cycle"]].duplicated().sum())

Merged BMX: (28061, 79)
Merged GHB: (20124, 80)
Merged GLU: (9556, 85)
Merged BPX: (9556, 107)
Merged BIOPRO: (9556, 147)
Merged DIQ: (9556, 200)
Merged KIQ_U: (7807, 216)
Merged SMQ: (7807, 257)
Merged ALQ: (7807, 273)
Merged PAQ: (7807, 368)
Merged DBQ: (7807, 422)
Merged MCQ: (7807, 553)
Merged BPQ: (7807, 566)
Merged CBC: (7807, 587)
Merged HDL: (7807, 589)
Merged TCHOL: (7807, 591)
Merged TRIGLY: (7807, 600)
Merged INS: (7807, 606)
Merged HSCRP: (5139, 608)
Merged ALB_CR: (5139, 615)
Merged HSQ: (5139, 623)

Final master shape: (5139, 623)
Duplicate SEQN+cycle: 0


In [4]:
# eGFR using CKD-EPI 2021 (race-free)
def egfr_ckdepi_2021(serum_creatinine, age, sex):
    scr = serum_creatinine
    kappa = np.where(sex == 1, 0.9, 0.7)
    alpha = np.where(sex == 1, -0.302, -0.241)
    ratio = scr / kappa
    ratio_min = np.minimum(ratio, 1)
    ratio_max = np.maximum(ratio, 1)
    egfr = 142 * (ratio_min ** alpha) * (ratio_max ** -1.200) \
           * (0.9938 ** age) \
           * np.where(sex == 1, 1.0, 1.012)
    return egfr

df["eGFR"] = egfr_ckdepi_2021(
    serum_creatinine=df["LBXSCR"],
    age=df["RIDAGEYR"],
    sex=df["RIAGENDR"]
)

# Mean blood pressure from available readings
bp_sys_cols = ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]
bp_dia_cols = ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]

df["mean_sbp"] = df[bp_sys_cols].mean(axis=1, skipna=True)
df["mean_dbp"] = df[bp_dia_cols].mean(axis=1, skipna=True)

C:\Users\MiR\AppData\Local\Temp\ipykernel_17464\547107233.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["eGFR"] = egfr_ckdepi_2021(
C:\Users\MiR\AppData\Local\Temp\ipykernel_17464\547107233.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["mean_sbp"] = df[bp_sys_cols].mean(axis=1, skipna=True)
C:\Users\MiR\AppData\Local\Temp\ipykernel_17464\547107233.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

In [5]:
def label_diabetes(row):
    if row["RIDEXPRG"] == 1:
        return np.nan
    if pd.isna(row["LBXGH"]) and pd.isna(row["LBXGLU"]) and pd.isna(row["DIQ010"]):
        return np.nan
    if (row["LBXGH"] >= 6.5) or (row["LBXGLU"] >= 126) or (row["DIQ010"] == 1):
        return 1
    return 0

def label_hypertension(row):
    if pd.isna(row["mean_sbp"]) and pd.isna(row["mean_dbp"]) and pd.isna(row["BPQ020"]):
        return np.nan
    if (row["mean_sbp"] >= 130) or (row["mean_dbp"] >= 80) or (row["BPQ020"] == 1):
        return 1
    return 0

def label_ckd(row):
    if pd.isna(row["eGFR"]) and pd.isna(row["KIQ025"]) and pd.isna(row["KIQ022"]):
        return np.nan
    if (row["eGFR"] < 60) or (row["KIQ025"] == 1) or (row["KIQ022"] == 1):
        return 1
    return 0

def label_dyslipidemia(row):
    if pd.isna(row["LBXTC"]) and pd.isna(row["LBDHDD"]) and pd.isna(row["LBXTR"]):
        return np.nan
    hdl_low = (row["LBDHDD"] < 40) if row["RIAGENDR"] == 1 else (row["LBDHDD"] < 50)
    if (row["LBXTC"] >= 200) or hdl_low or (row["LBXTR"] >= 150):
        return 1
    return 0

def label_cvd(row):
    cols = ["MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F"]
    if row[cols].isna().all():
        return np.nan
    if (row[cols] == 1).any():
        return 1
    return 0

def label_metabolic_syndrome(row):
    components = 0
    if (row["RIAGENDR"] == 1 and row["BMXWAIST"] > 102) or (row["RIAGENDR"] == 2 and row["BMXWAIST"] > 88):
        components += 1
    if row["LBXTR"] >= 150:
        components += 1
    hdl_low = (row["LBDHDD"] < 40) if row["RIAGENDR"] == 1 else (row["LBDHDD"] < 50)
    if hdl_low:
        components += 1
    if not pd.isna(row["mean_sbp"]) and not pd.isna(row["mean_dbp"]):
        if row["mean_sbp"] >= 130 or row["mean_dbp"] >= 85:
            components += 1
    if row["LBXGLU"] >= 100:
        components += 1
    if pd.isna(row["BMXWAIST"]) or pd.isna(row["LBXTR"]) or pd.isna(row["LBDHDD"]) \
       or pd.isna(row["mean_sbp"]) or pd.isna(row["LBXGLU"]):
        return np.nan
    return 1 if components >= 3 else 0

def label_liver_disease(row):
    if pd.isna(row["LBXSATSI"]) and pd.isna(row["LBXSASSI"]) and pd.isna(row["LBXSGTSI"]) and pd.isna(row["BMXBMI"]):
        return np.nan
    alt_uln = 40 if row["RIAGENDR"] == 1 else 30
    ast_uln = 40
    ggt_uln = 50
    enzyme_elevated = (row["LBXSATSI"] > alt_uln) | (row["LBXSASSI"] > ast_uln) | (row["LBXSGTSI"] > ggt_uln)
    bmi_high = row["BMXBMI"] >= 25
    if enzyme_elevated and bmi_high:
        return 1
    return 0

In [6]:
df["label_diabetes"] = df.apply(label_diabetes, axis=1)
df["label_hypertension"] = df.apply(label_hypertension, axis=1)
df["label_ckd"] = df.apply(label_ckd, axis=1)
df["label_dyslipidemia"] = df.apply(label_dyslipidemia, axis=1)
df["label_cvd"] = df.apply(label_cvd, axis=1)
df["label_metabolic_syndrome"] = df.apply(label_metabolic_syndrome, axis=1)
df["label_liver_disease"] = df.apply(label_liver_disease, axis=1)

C:\Users\MiR\AppData\Local\Temp\ipykernel_17464\1474595990.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["label_diabetes"] = df.apply(label_diabetes, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_17464\1474595990.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["label_hypertension"] = df.apply(label_hypertension, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_17464\1474595990.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

In [7]:
label_cols = [c for c in df.columns if c.startswith("label_")]
print("Label counts (combined):")
for col in label_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

Label counts (combined):

label_diabetes:
label_diabetes
0.0    3949
1.0    1133
NaN      57
Name: count, dtype: int64

label_hypertension:
label_hypertension
1    2893
0    2246
Name: count, dtype: int64

label_ckd:
label_ckd
0    4666
1     473
Name: count, dtype: int64

label_dyslipidemia:
label_dyslipidemia
1.0    2844
0.0    1993
NaN     302
Name: count, dtype: int64

label_cvd:
label_cvd
0    4506
1     633
Name: count, dtype: int64

label_metabolic_syndrome:
label_metabolic_syndrome
0.0    2702
1.0    1613
NaN     824
Name: count, dtype: int64

label_liver_disease:
label_liver_disease
0.0    4396
1.0     730
NaN      13
Name: count, dtype: int64


In [8]:
output_path = os.path.join(PROCESSED_DIR, "labeled_master_combined.csv")
df.to_csv(output_path, index=False)
print(f"Combined labeled master saved to {output_path}")
print(f"Shape: {df.shape}")

Combined labeled master saved to ../data/processed\labeled_master_combined.csv
Shape: (5139, 633)
